# Laboratorio 2.3 — Primeros pasos con PySpark

Este notebook está guiado: todo el código de las transformaciones ya está escrito y probado. Tu trabajo es **ejecutar las celdas en orden** y completar las celdas marcadas con `✏️`.

**Antes de empezar:** sube el fichero `tienda_online_ventas.csv` a esta sesión de Colab (icono de carpeta en la barra lateral izquierda → botón de subir), en el mismo directorio desde el que ejecutas el notebook.

## 0. Instalación de PySpark y arranque de la sesión

La siguiente celda instala PySpark. Necesita conexión a internet — Colab ya la proporciona por defecto. La instalación puede tardar 1-2 minutos.

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Lab23").getOrCreate()
print("SparkSession creada. Versión de Spark:", spark.version)

## 1. Carga del CSV

Cargamos el mismo dataset que ya conoces del laboratorio 2.1: `tienda_online_ventas.csv`. `inferSchema=True` le pide a Spark que detecte automáticamente el tipo de cada columna (equivalente a lo que pandas hace por defecto al leer un CSV).

In [ ]:
df = spark.read.csv("tienda_online_ventas.csv", header=True, inferSchema=True)

print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

In [ ]:
df.show(5)

## 2. Cinco transformaciones básicas

### Transformación 1 — Filtrar filas: pedidos de categoría "Electrónica"

En Sheets, esto sería aplicar un filtro a la columna `categoria` y quedarte solo con las filas donde el valor es "Electrónica". En Spark se hace con `.filter()` (o su alias `.where()`), pasando una condición sobre una columna con `F.col(...)`.

In [ ]:
df_electronica = df.filter(F.col("categoria") == "Electrónica")

print(f"Pedidos de categoría Electrónica: {df_electronica.count()}")
df_electronica.show(5)

### Transformación 2 — Seleccionar columnas

Equivalente a ocultar columnas en Sheets, o a un `SELECT columna1, columna2 FROM tabla` en SQL. `.select()` se queda solo con las columnas indicadas, descartando el resto.

In [ ]:
df_resumen = df.select("pedido_id", "fecha", "categoria", "importe")
df_resumen.show(5)

### Transformación 3 — Agrupar y agregar: importe medio y total por categoría

Equivalente a una tabla dinámica en Sheets, agrupando por `categoria` y calculando el promedio y la suma de `importe`. En Spark se combina `.groupBy()` con `.agg()`, usando funciones de `pyspark.sql.functions` (importado como `F`) para cada agregación.

In [ ]:
df_por_categoria = (
    df.groupBy("categoria")
    .agg(
        F.avg("importe").alias("importe_medio"),
        F.sum("importe").alias("importe_total"),
        F.count("*").alias("num_pedidos"),
    )
)

df_por_categoria.show()

### Transformación 4 — Ordenar resultados

Equivalente a ordenar una columna de mayor a menor en Sheets. En Spark se usa `.orderBy()`, combinado con `F.desc()` para indicar orden descendente.

In [ ]:
df_por_categoria_ordenado = df_por_categoria.orderBy(F.desc("importe_total"))
df_por_categoria_ordenado.show()

### Transformación 5 — Combinar con `join`: añadir el margen comercial estimado

Equivalente a un `BUSCARV`/`VLOOKUP` en Sheets, donde traes una columna de otra tabla haciendo coincidir una clave común (aquí, `categoria`). En Spark se hace con `.join()`, indicando la columna clave (`on=`) y el tipo de unión (`how=`).

Para este ejercicio creamos una pequeña tabla de referencia con el margen comercial estimado (%) de cada categoría — datos ficticios, a modo de ejemplo de una segunda fuente de datos que habría que combinar con las ventas.

In [ ]:
margenes = spark.createDataFrame(
    [
        ("Electrónica", 12.5),
        ("Hogar", 22.0),
        ("Deporte", 18.0),
        ("Moda", 35.0),
        ("Papelería", 28.0),
    ],
    ["categoria", "margen_pct"],
)

df_con_margen = df_por_categoria_ordenado.join(margenes, on="categoria", how="left")
df_con_margen = df_con_margen.withColumn(
    "beneficio_estimado",
    F.round(F.col("importe_total") * F.col("margen_pct") / 100, 2),
)

df_con_margen.orderBy(F.desc("beneficio_estimado")).show()

## 3. Lazy evaluation: el plan antes que la ejecución

Spark no ejecuta cada transformación en el momento en que la escribes. En su lugar, va acumulando un **plan lógico** con todas las transformaciones encadenadas (`filter`, `select`, `groupBy`...) y solo lo optimiza y ejecuta de verdad cuando le pides un resultado concreto — una **acción** como `.show()`, `.collect()` o `.count()`. A esto se le llama evaluación perezosa (*lazy evaluation*).

Vamos a pedirle a Spark el plan de ejecución de la consulta de la Transformación 3 (agrupar por categoría) con `.explain()`.

In [ ]:
df_por_categoria.explain()

### ✏️ Tu observación

Lee el plan impreso arriba. Suele leerse de abajo hacia arriba para seguir el orden real de ejecución: primero se lee el fichero, y las operaciones se van aplicando hacia arriba.

Identifica en el plan al menos dos etapas (por ejemplo, `FileScan`, `HashAggregate`, `Exchange`) y anota qué crees que hace cada una:

- Etapa 1: ____________________ → hace: ____________________
- Etapa 2: ____________________ → hace: ____________________

¿En qué se parece este plan a cómo resolverías tú mentalmente la misma consulta paso a paso (leer los datos, agrupar, calcular el promedio y la suma)? ¿En qué se diferencia? (pista: la palabra `Exchange` en el plan indica que Spark tiene que redistribuir datos entre particiones para poder agrupar por `categoria` — es justo lo que el apunte de procesamiento llama *shuffle*, la operación más costosa de un job distribuido).

_(tu respuesta aquí)_

## 4. Extra opcional — la misma consulta en SQL

Spark permite registrar un DataFrame como una vista temporal y consultarlo con SQL estándar, mezclando ambos estilos según convenga. Esta celda es opcional: reproduce la Transformación 3 (agrupar por categoría) pero escrita en SQL en lugar de con la API de DataFrame, para que compruebes que el resultado es idéntico.

In [ ]:
df.createOrReplaceTempView("ventas")

resultado_sql = spark.sql('''
    SELECT categoria,
           AVG(importe) AS importe_medio,
           SUM(importe) AS importe_total,
           COUNT(*) AS num_pedidos
    FROM ventas
    GROUP BY categoria
    ORDER BY importe_total DESC
''')

resultado_sql.show()

## 5. Tabla comparativa: PySpark ↔ hoja de cálculo

Completa la columna de la derecha con el equivalente en Sheets/Excel de cada operación de PySpark que has usado en este notebook.

| Operación en PySpark | Equivalente en Sheets/Excel |
| --- | --- |
| `df.filter(F.col("categoria") == "Electrónica")` | _(completar)_ |
| `df.select("pedido_id", "fecha", "categoria", "importe")` | _(completar)_ |
| `df.groupBy("categoria").agg(F.avg("importe"), F.sum("importe"))` | _(completar)_ |
| `df.orderBy(F.desc("importe_total"))` | _(completar)_ |
| `df_por_categoria.join(margenes, on="categoria", how="left")` | _(completar)_ |

## Cierre

Guarda el notebook con todas las celdas ejecutadas (Archivo → Descargar → Descargar .ipynb, o simplemente deja el historial de ejecución visible) y entrégalo junto con la tabla comparativa y tu observación sobre el plan de ejecución.